# Exploring the K10plus Biblical Studies harvest

Opening cells for the Parquet files produced by `biblical-books harvest` + `biblical-books export`.

**Read this before drawing conclusions from any number below.**

- `records.parquet` is the union of the nine configured searches (DDC `22*`, LCC `BS*`, four subject
  searches, three title supplements), deduplicated by K10plus record ID. It is a set of **candidates**.
- `books.parquet` is a **format filter** on that set: MARC leader type `a` + bibliographic level `m`
  (textual monographs, edited volumes included). It still contains primary texts, devotional works and
  false positives. It is *not* a validated academic corpus.
- Coverage of K10plus is incomplete, so "the configured union" is not "all Biblical Studies books".
- The catalogue is live and was paged by offset; counts can drift while harvesting. `export.json`
  records `selection_complete`; a partial export is marked as such and may be used for inspection only.
- Print/e-book editions, translations and revised editions are separate rows. Nothing is collapsed at
  work level here.

Analysis conventions used below: `publication_year` comes from MARC 008 date1 when numeric, otherwise
from a single unambiguous year in 260/264 $c; `date_type` says what date1 means (reprint, multipart,
forthcoming…). Unknown stays null — never impute it.

## 1. Manifests: what was exported, and was the harvest complete?

`export.json` is written by `biblical-books export`; it repeats the `status` report so a figure can
always be traced to a known retrieval date and completion state.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# Point K10PLUS_DATA at a frozen copy (e.g. an archived export) to make a notebook run reproducible.
DATA_DIR = Path(os.environ.get("K10PLUS_DATA", "data/k10plus"))

manifest = json.loads((DATA_DIR / "export.json").read_text())
print("exported_at       :", manifest["exported_at"])
print("selection_complete:", manifest["status"]["selection_complete"])
print("records           :", manifest["records"])
print("textual monographs:", manifest["textual_monographs"])

per_query = pd.DataFrame(manifest["status"]["queries"])
per_query[["id", "cql", "pages", "downloaded_occurrences", "advertised_total", "complete"]]

## 2. Load the tables

Load **only the columns you need**: `marc_json` dominates memory (the whole MARC record as JSON per row).
Both files are streamed into Parquet in bounded row groups, so reading a column subset is cheap.

In [ ]:
COLUMNS = [
    "record_id", "full_title", "publication_year", "date_type", "languages",
    "record_type", "bibliographic_level", "is_textual_monograph", "matched_queries",
]

books = pd.read_parquet(DATA_DIR / "books.parquet", columns=COLUMNS, dtype_backend="pyarrow")
candidates = pd.read_parquet(
    DATA_DIR / "records.parquet",
    columns=["record_id", "is_textual_monograph", "matched_queries"],
    dtype_backend="pyarrow",
)

assert books.record_id.is_unique, "record_id must be unique"
assert candidates.record_id.is_unique, "record_id must be unique"
assert len(books) == manifest["textual_monographs"], "row count disagrees with export.json"
assert len(candidates) == manifest["records"], "row count disagrees with export.json"

print(f"candidates: {len(candidates):,}   textual monographs: {len(books):,}")
print(f"monograph share: {len(books) / len(candidates):.1%}")
books.head(5)

## 3. Which search found what?

Advertised counts overlap heavily, so the useful question is not "how many per query" but
**"how many records would be lost if this query were dropped"** — i.e. records matched by that query
and nothing else.

In [ ]:
long = candidates[["record_id", "matched_queries"]].explode("matched_queries")
long = long.rename(columns={"matched_queries": "query"})

hits = long.groupby("query").size().rename("records_matched")
exclusive = (
    candidates.assign(n=candidates.matched_queries.list.len())
    .query("n == 1")
    .matched_queries.explode()
    .value_counts()
    .rename("only_source_for")
)
contribution = pd.concat([hits, exclusive], axis=1).fillna(0).astype(int)
contribution["share_of_candidates"] = contribution.records_matched / len(candidates)
contribution.sort_values("only_source_for", ascending=False)

## 4. Publication years

Missing years are reported, not filled. `date_type` (MARC 008/06) explains what date1 means:
`s` single known date, `m` multiple dates (multipart), `r` reprint, `t` publication + copyright.
Forthcoming years are legitimate values.

In [ ]:
years = books.publication_year.dropna().astype("int64")
print(f"known year: {len(years):,} of {len(books):,} ({len(years) / len(books):.1%})")
print("range     :", years.min(), "-", years.max())
print()
print("date_type:")
print(books.date_type.value_counts(dropna=False).head(8).to_string())
print()

decade = (years // 10 * 10).value_counts().sort_index()
decade[decade.index >= 1800]

In [ ]:
# Optional plot; matplotlib is a notebook-only dependency, not part of the project.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed - skipping plot")
else:
    yearly = years[(years >= 1950) & (years <= years.max())].value_counts().sort_index()
    ax = yearly.rolling(3, center=True, min_periods=1).mean().plot(
        figsize=(11, 3.5), title="Textual monographs per publication year (3-year mean)"
    )
    ax.set_xlabel("publication_year")
    ax.set_ylabel("records")
    plt.tight_layout()

## 5. Languages

Codes are preserved as catalogued (MARC 041 $a plus 008/35-37), never translated, and never normalised
across `eng`/`en`, `ger`/`deu`, `fre`/`fra` — that harmonisation belongs in the analysis, explicitly.

In [ ]:
langs = books.languages.explode().dropna()
n_lang = books.languages.apply(len)

print(f"records with no language code: {(n_lang == 0).sum():,} ({(n_lang == 0).mean():.1%})")
print(f"records with >1 language    : {(n_lang > 1).sum():,}")
print()
langs.value_counts().head(15)

## 6. Missingness of the key metadata

Cataloguing depth varies by record; a low fill rate is a statement about K10plus editing, not about the
books. `is_textual_monograph` and `leader` are always present by construction.

In [ ]:
CHECK = [
    "title", "subtitle", "responsibility", "publication_year", "date_text",
    "publishers", "publication_places", "isbns", "languages", "subjects",
    "series", "ddc", "lcc", "abstract", "urls",
]
probe = pd.read_parquet(DATA_DIR / "books.parquet", columns=CHECK, dtype_backend="pyarrow")

def filled(series):
    """True when a scalar is non-null/non-empty, or a list column has at least one element."""
    if isinstance(series.dtype, pd.ArrowDtype) and str(series.dtype).startswith("list"):
        return series.list.len().fillna(0) > 0
    if series.dtype == object or str(series.dtype).startswith("string"):
        return series.notna() & (series.astype("string").str.strip() != "")
    return series.notna()

pd.DataFrame({"filled": {c: int(filled(probe[c]).sum()) for c in CHECK}, "missing_pct": {c: round(100 * (~filled(probe[c])).mean(), 1) for c in CHECK}})

## 7. Publishers, subjects, series

Repeated metadata are genuine list columns, so explode them rather than splitting strings. Publisher
strings keep catalogue punctuation and abbreviations; treat these counts as a rough orientation.

In [ ]:
extra = pd.read_parquet(
    DATA_DIR / "books.parquet",
    columns=["publishers", "subjects", "series", "ddc", "lcc"],
    dtype_backend="pyarrow",
)

for column in ["publishers", "subjects", "series"]:
    print(f"--- top 10 {column}")
    print(extra[column].explode().dropna().value_counts().head(10).to_string())
    print()

## 8. Spot-check relevance by match type

Records reached only by a classification, only by a subject heading, or only by a title keyword are
systematically different kinds of evidence — and the title supplements are the known risk for false
positives. Sample them separately instead of judging the corpus from its front page.

In [ ]:
CLASS = {"ddc_bible", "lcc_bible"}
SUBJECT = {"subject_bibel", "subject_bible", "subject_old_testament", "subject_new_testament"}
TITLE = {"title_biblical", "title_biblique", "title_biblisch"}

def match_kind(queries):
    kinds = {
        "classification" if q in CLASS else "subject" if q in SUBJECT else "title"
        for q in queries
    }
    return "+".join(sorted(kinds)) if len(kinds) > 1 else next(iter(kinds))

books = books.assign(match_kind=books.matched_queries.apply(match_kind))
books.match_kind.value_counts().to_frame("records")

In [ ]:
for kind in ["classification", "subject", "title"]:
    subset = books[books.match_kind == kind]
    print(f"=== {kind}-only: {len(subset):,} records")
    sample = subset.sample(min(6, len(subset)), random_state=0)
    for row in sample.itertuples():
        langs = ", ".join(row.languages) if isinstance(row.languages, list) else ""
        print(f"  {row.publication_year or '????'}  {row.full_title[:110]}  [{langs}]")
    print()

## 9. The original research scope (a later filter, not a harvest restriction)

The harvest deliberately keeps everything. The historical question was about title vocabulary in
English, German and French, 2000–2025; applying it is a one-line filter, and the loss it causes is
visible in the counts.

In [ ]:
TARGET_LANGUAGES = {"eng", "en", "ger", "deu", "de", "fre", "fra", "fr"}

in_window = books.publication_year.between(2000, 2025).fillna(False)
in_language = books.languages.apply(lambda xs: bool(TARGET_LANGUAGES.intersection(xs or [])))
scope = books[in_window & in_language]

print(f"all textual monographs      : {len(books):,}")
print(f"published 2000-2025         : {int(in_window.sum()):,}")
print(f"... also eng/ger/fre        : {len(scope):,}")
print()
scope.languages.explode().value_counts().head(8)

## 10. Provenance: never analyse only the parsed columns

Every row keeps the full returned MARC in `marc_json`, and the original response bytes stay in
`raw/<query>/<start>.xml.gz` with a SHA-256 per page in the `pages` table of `harvest.sqlite`. When a
parsed value looks wrong, go back to `marc_json` before touching the pipeline — and never hand-edit a
parsed table, because `export` re-emits the payloads stored in SQLite, it does not re-parse the XML.

In [ ]:
import sqlite3

one = pd.read_parquet(
    DATA_DIR / "books.parquet",
    columns=["record_id", "full_title", "marc_json", "first_seen"],
    dtype_backend="pyarrow",
).iloc[0]

marc = json.loads(one.marc_json)
print("record_id :", one.record_id, "| retrieved:", one.first_seen)
print("title     :", one.full_title)
print("leader    :", marc["leader"])
for field in marc["fields"][:6]:
    print(" ", field)

print()
with sqlite3.connect(f"file:{DATA_DIR / 'harvest.sqlite'}?mode=ro", uri=True) as db:
    print("pages and raw responses for this query, one row each:")
    print(pd.read_sql("SELECT query_id, start, count, total, sha256, raw_path FROM pages LIMIT 3", db).to_string(index=False))